# Demo: Architettura RAG per il CNI

**Tesi:** Architettura RAG per l'estrazione e la consultazione intelligente dei dati pubblici del Consiglio Nazionale degli Ingegneri

Questa demo mostra:
1. Il flusso RAG completo (crawl → chunk → embed → indicizza → query)
2. L'orchestrazione con LangGraph
3. L'impatto del reranker
4. Esempi di query reali sul CNI

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import time
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel

console = Console()
print('Setup completato ✅')

## 1. Panoramica dell'Architettura

In [ ]:
console.print(Panel.fit("""
[bold cyan]Architettura del Sistema RAG[/bold cyan]

┌─────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  Crawler    │───>│   Cleaner    │───>│   Chunker    │───>│  Embedder    │
│  (httpx+BS) │    │  (trafilatura)│   │  (langchain) │    │ (MiniLM-L6)  │
└─────────────┘    └──────────────┘    └──────────────┘    └──────┬───────┘
                                                                   │
                                                         ┌─────────▼────────┐
                                                         │  Qdrant Vector   │
                                                         │     Store        │
                                                         └─────────┬────────┘
                                                                   │
┌─────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────┴───────┐
│  Angular FE │<───│  FastAPI     │<───│  LangGraph   │<───│  Retriever   │
│  (localhost  │    │  (REST/SSE)  │    │  Orchestrator│    │  (Hybrid)    │
│    :4200)    │    └──────────────┘    └──────┬───────┘    └──────────────┘
└─────────────┘                                 │
                                         ┌──────▼───────┐
                                         │  LLM (Llama) │
                                         │  LM Studio   │
                                         └──────────────┘
""", title="Architettura", border_style="blue"))

## 2. Componenti del Sistema

### Moduli Python creati:

In [ ]:
modules = [
    ("Ingestion", "Crawler, Parser, Cleaner, Chunker, Embedder"),
    ("Vector Store", "Qdrant Client, Indexer, Retriever"),
    ("RAG Orchestration", "Query Classifier, Hybrid Retriever, Reranker, Prompt Builder, RAG Chain (LangGraph)"),
    ("Inference", "LLM Client (LM Studio), Response Generator, Citation Builder"),
    ("Governance", "PII Filter, Public Data Filter, Quality Check, Monitoring"),
    ("API", "FastAPI endpoints: /query, /query/stream, /ingest, /health"),
    ("Frontend", "Angular 18 con chat interattiva"),
]

for name, desc in modules:
    console.print(f"[bold yellow]{name}[/bold yellow]")
    console.print(f"  {desc}\n")

## 3. Flusso RAG con LangGraph

Il cuore del sistema è il grafo LangGraph che orchestra il flusso:

In [ ]:
console.print(Panel.fit("""
[bold]Grafo LangGraph - RAG Chain[/bold]

  [cyan]classify[/cyan] ──► [cyan]retrieve[/cyan] ──► [cyan]rerank[/cyan] ──► [cyan]build_prompt[/cyan] ──► [cyan]generate[/cyan] ──► [cyan]build_citations[/cyan] ──► [green]END[/green]
       │               │               │               │               │               │
       ▼               ▼               ▼               ▼               ▼               ▼
  QueryClass.     HybridRetr.     Reranker       PromptBuild.    ResponseGen.    CitationB.
  (categorizza)   (ricerca        (riordina      (costruisce     (LLM genera     (estrae
                   vettoriale)     risultati)      prompt)         risposta)       fonti)
""", title="LangGraph Flow", border_style="green"))

## 4. Query Esempio

Esegue una query sul sistema RAG (richiede LM Studio in esecuzione):

In [ ]:
from src.core.model_factory import ModelFactory
from src.rag.rag_chain import RAGChain
from src.rag.query_classifier import QueryClassifier
from src.rag.hybrid_retriever import HybridRetriever
from src.core.config_loader import ConfigLoader

# Esempio di sola retrieval (non serve LLM)
qc = QueryClassifier()
embeddings = ModelFactory.create_embeddings()
retriever = HybridRetriever(embeddings)

test_queries = [
    "Quali sono gli organi del Consiglio Nazionale degli Ingegneri?",
    "Come funziona la formazione continua per gli ingegneri?",
    "Quali commissioni esistono presso il CNI?",
    "Cosa dice il codice deontologico degli ingegneri?",
]

for q in test_queries:
    category = qc.classify(q)
    console.print(f"[bold]Q:[/bold] {q}")
    console.print(f"[dim]Categoria rilevata:[/dim] [cyan]{category}[/cyan]")
    
    # Prova retrieval
    try:
        results = retriever.retrieve(q, top_k=3)
        if results:
            console.print(f"[green]✓[/green] Recuperati {len(results)} documenti")
            for i, r in enumerate(results, 1):
                console.print(f"  {i}. [blue]{r.get('title', 'N/D')}[/blue] (score: {r.get('score', 0):.3f})")
        else:
            console.print("[yellow]⚠ Nessun documento recuperato (eseguire ingestion prima)[/yellow]")
    except Exception as e:
        console.print(f"[red]✗ Errore: {e}[/red]")
    console.print()

## 5. Confronto: Con vs Senza Reranker

In [ ]:
from src.rag.reranker import Reranker

console.print("[bold]Test Reranker[/bold]\n")

# Simula risultati grezzi
mock_results = [
    {"title": "Pagina generica", "content": "test " * 100, "score": 0.92},
    {"title": "Documento normativa", "content": "normativa legge regolamento " * 100, "score": 0.88},
    {"title": "News evento", "content": "evento notizia comunicato " * 100, "score": 0.85},
    {"title": "Codice deontologico", "content": "codice deontologico ingegnere " * 100, "score": 0.82},
    {"title": "Contatti sede", "content": "sede telefono email contatti " * 100, "score": 0.80},
]

reranker = Reranker()
query = "Qual è il codice deontologico degli ingegneri?"

console.print(f"Query: [italic]{query}[/italic]\n")
console.print("[yellow]Prima del reranking (ordinato per score vettoriale):[/yellow]")
for i, r in enumerate(mock_results, 1):
    console.print(f"  {i}. {r['title']} (score: {r['score']:.3f})")

console.print()

if reranker.enabled:
    reranked = reranker.rerank(query, mock_results)
    console.print("[green]Dopo il reranking:[/green]")
    for i, r in enumerate(reranked, 1):
        score = r.get('rerank_score', r.get('score', 0))
        console.print(f"  {i}. {r['title']} (score: {score:.3f})")
else:
    console.print("[yellow]Cross-encoder non disponibile, skip reranking[/yellow]")

## 6. Metriche di Valutazione

Lo script di benchmark calcola:

In [ ]:
console.print(Panel.fit("""
[bold]Metriche implementate:[/bold]

  [cyan]MRR[/cyan] (Mean Reciprocal Rank)
    - Media del reciproco del rango del primo documento rilevante
    - Indica quanto "in alto" appare il primo risultato utile

  [cyan]Recall@k[/cyan] (k=1, 3, 5)
    - Frazione di documenti rilevanti trovati nei top-k
    - Misura la completezza del recupero

  [cyan]Precision@k[/cyan] (k=1, 3)
    - Frazione di documenti rilevanti tra i top-k recuperati
    - Misura la pertinenza dei risultati

  [cyan]Classification Accuracy[/cyan]
    - Percentuale di query classificate nella categoria corretta

  [cyan]Latenza Media[/cyan]
    - Tempo medio di risposta in millisecondi
""", title="Metriche", border_style="blue"))

## 7. Come Eseguire il Progetto Completo

In [ ]:
console.print("[bold]Setup Rapido[/bold]\n")

steps = [
    ("1", "Avvia LM Studio", "Carica un modello Llama (es. llama-3.2-3b-instruct) su http://localhost:1234"),
    ("2", "Installa dipendenze", "pip install -r requirements.txt"),
    ("3", "Configura .env", "cp .env.example .env"),
    ("4", "Crawl e indicizza", "python scripts/run_ingestion.py"),
    ("5", "Avvia API", "python scripts/run_api.py"),
    ("6", "Avvia Frontend", "cd frontend && ng serve"),
    ("7", "Benchmark", "python benchmarks/run_benchmark.py"),
]

for num, title, desc in steps:
    console.print(f"[cyan]{num}.[/cyan] [bold]{title}[/bold]")
    console.print(f"     {desc}")

In [ ]:
console.print("\n" + "="*60)
console.print("[bold green]Demo completata![/bold green]")
console.print("="*60)
console.print("""
Per qualsiasi domanda sul progetto:
- Codice: /src, /scripts, /frontend
- Benchmark: /benchmarks
- Configurazioni: /config
- Test: /tests
""")